# OSNet · аудит ошибок и устойчивости к анонимизации

Без обучения, изменений MVP или исходных изображений. Сначала полный outer-validation отчёт ошибок, затем **отдельное парное сравнение на случайной небольшой подвыборке** с точными ручными масками.

Для масок нужно участие человека: Run All создаст HTML-разметчик, но не станет выдавать неразмеченные зоны за проверенные. CLIP-ReID можно обучать независимо в соседнем notebook.

In [ ]:
from pathlib import Path
import os, sys, json
candidates = [Path.cwd(), *Path.cwd().parents, Path.cwd() / "Car-classification-MSK"]
ROOT = next((p for p in candidates if (p / "backend/core.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Открой notebook внутри Car-classification-MSK")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("Repository:", ROOT)


In [ ]:
from IPython.display import display
import ipywidgets as widgets
from training.pipeline import ensure_splits
from training.audit import error_audit, prepare_mask_audit, save_annotations, paired_mask_audit
OUT = ROOT / "OSNet-AIN-x1.0/audit_07_errors_masks/results"
rows, split = ensure_splits()


## 1. Ошибки рабочего OSNet

Фиксированы текущие веса, evaluator, порог и streaming reranking 20/3/0.5. В errors.html показаны query → top-5 → пример cross-camera positive. В errors.json — все ошибочные решения и размеры кропов. Отказы известным авто и принятие неизвестных показаны отдельно.

In [ ]:
report = error_audit(rows, split, OUT)
print(json.dumps({k: v for k, v in report.items() if k != "errors"}, ensure_ascii=False, indent=2))
print("Открой HTML локально:", OUT / "errors.html")


## 2. Случайная подвыборка для точных масок

По умолчанию 32 identity из outer validation, 32 query + 94 gallery (126 кропов на текущем наборе). Отбор не зависит от ошибок модели. Можно выбрать весь validation (`identities=None`), но тогда используй новую папку OUT, чтобы не потерять разметку.

У маленькой галереи другая сложность: абсолютные метрики нельзя сравнивать с общими 81,47%. Нас интересует парное изменение до/после при неизменных query/gallery.

In [ ]:
plan = prepare_mask_audit(rows, split, OUT, identities=32)
print("Нужно проверить кропов:", len(plan["images"]))
print("Открой этот файл в обычном браузере:", OUT / "annotate_masks.html")


## 3. Разметка

Обведи уже размытые номерные области/лица внутри каждого кропа и нажми «Проверено». Если анонимизированных областей нет, просто отметь проверку. Не маскируй кузов наугад. После всех кадров скачай masks.json. Промежуточный файл можно загрузить обратно в HTML-разметчик.

Загрузи окончательный JSON виджетом ниже; затем выполни следующую ячейку. Либо положи masks.json вручную в OUT.

In [ ]:
upload = widgets.FileUpload(accept=".json", multiple=False, description="masks.json")
display(upload)


## 4. Парная проверка

Проверяются 100% кропов выбранного протокола, включая gallery; неразмеченные/изменённые файлы блокируют оценку. Непрозрачная чёрная заливка применяется до resize. Веса, порог и параметры reranking одинаковы до/после; порог не перекалибруется под маски.

In [ ]:
annotation_path = OUT / "masks.json"
if upload.value:
    annotations = save_annotations(upload.value[0]["content"], plan, annotation_path)
elif annotation_path.exists():
    annotations = json.loads(annotation_path.read_text())
else:
    annotations = None
    print("Ожидается ручная разметка. Аудит устойчивости пока НЕ выполнен.")
if annotations is not None:
    result = paired_mask_audit(rows, plan, annotations, OUT)
    print(json.dumps(result, ensure_ascii=False, indent=2))


## Как читать результат

`delta_mAP_at_10` — after − before; отрицательное значение означает падение. Также сравниваются F1/TNR при фиксированном пороге, изменившиеся top-1 и cosine эмбеддингов для замаскированных кропов. Маленькая ручная выборка не доказывает независимость от номера и не заменяет контроль организаторов. Сами по себе attention maps и нижние proxy-маски тоже не являются таким доказательством.